# Week 3 Data Pipeline: Gig Workers Rights Assistant

Kaveetiran A/L Sitambaram (A24AI0038), SAIA 3113 AI Engineering

Pathway A (PDF / text). Sources are the official Gig Workers Act 2025 documents from the KESUMA portal.

Requirements this data supports: FR1, NFR1 (legal text), FR2, NFR2, RAI4 (section citation), NFR4 (version and checksum).

## Part A: Start with the Data

### Cell 1: Workspace setup (run first)

In [1]:
# Cell 1: Workspace setup (RUN FIRST)
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
BASE_DIR = Path("/content/week3_project") if IN_COLAB else Path("week3_project").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW, PROCESSED, BASE, OUT = RAW_DIR, PROCESSED_DIR, BASE_DIR, PROCESSED_DIR
print("Raw folder:", RAW_DIR.relative_to(BASE_DIR.parent).as_posix())
print("Processed folder:", PROCESSED_DIR.relative_to(BASE_DIR.parent).as_posix())

Raw folder: week3_project/data/raw
Processed folder: week3_project/data/processed


### Cell 2: Load the source PDFs

Not using the dummy dataset. The KESUMA website times out from Colab, so the three PDFs are downloaded in the browser from the official links and uploaded to `week3_project/data/raw`. If a file is not there, the cell tries to download it.

| ID | Document | Use |
|---|---|---|
| S01 | Gig Workers Act 2025 (Act 872), English | Main legal text |
| S02 | P.U.(A) 143/2026 Social Security Regulations | 1.25% contribution rate |
| S03 | KESUMA FAQ (BM) | Inspected only for now |

In [2]:
# Cell 2: Load source PDFs
import hashlib
import json
import urllib.request
from datetime import datetime, timezone

PORTAL = "https://www.mohr.gov.my/aktapekerjagig2025/assets/documents/"
SOURCES = {
    "S01": {"file": "act872_en.pdf", "url": PORTAL + "Act%20872.pdf",
            "portal_name": "Act 872.pdf",
            "title": "Gig Workers Act 2025 (Act 872), English"},
    "S02": {"file": "pua143_2026.pdf",
            "url": PORTAL + "PUA143_2026%20-%20PERATURAN%20PEKERJA%20GIG%20(KESELAMATAN%20SOSIAL).pdf",
            "portal_name": "PUA143_2026 - PERATURAN PEKERJA GIG (KESELAMATAN SOSIAL).pdf",
            "title": "Gig Workers (Social Security) Regulations 2026, P.U.(A) 143/2026"},
    "S03": {"file": "kesuma_faq.pdf", "url": PORTAL + "faq.pdf",
            "portal_name": "faq.pdf",
            "title": "Soalan Lazim Akta Pekerja Gig 2025 (KESUMA FAQ)"},
}

def try_download(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (student project)"})
    for attempt in (1, 2):
        try:
            with urllib.request.urlopen(request, timeout=20) as response:
                return response.status, response.read()
        except Exception as error:
            last_error = error
    raise ConnectionError(last_error)

ingestion_log, missing = [], []
for sid, info in SOURCES.items():
    target = RAW_DIR / info["file"]
    original = RAW_DIR / info["portal_name"]
    if not target.is_file() and original.is_file():
        original.rename(target)
    if target.is_file():
        method, status = "uploaded to data/raw (downloaded from the official URL in a browser)", None
    else:
        try:
            status, data = try_download(info["url"])
            target.write_bytes(data)
            method = "downloaded directly from the official URL"
        except ConnectionError as error:
            missing.append(f"{info['portal_name']}  ({info['url']})")
            print(f"{sid}: not uploaded and direct download failed ({error})")
            continue
    data = target.read_bytes()
    if not data.startswith(b"%PDF"):
        raise ValueError(f"{sid}: {target.name} is not a PDF (maybe an error page was saved)")
    ingestion_log.append({
        "source_id": sid, "file": info["file"], "url": info["url"], "method": method,
        "http_status": status, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
        "ingested_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })
    print(f"{sid} {info['file']}: {len(data):,} bytes | {method}")

if missing:
    raise FileNotFoundError(
        "These sources are missing. Download them in a browser and upload them to "
        "week3_project/data/raw (Files panel), then run this cell again:\n  " + "\n  ".join(missing))

(BASE_DIR / "data" / "ingestion_log.json").write_text(json.dumps(ingestion_log, indent=2), encoding="utf-8")

S01 act872_en.pdf: 439,389 bytes | uploaded to data/raw (downloaded from the official URL in a browser)
S02 pua143_2026.pdf: 269,365 bytes | uploaded to data/raw (downloaded from the official URL in a browser)
S03 kesuma_faq.pdf: 686,646 bytes | uploaded to data/raw (downloaded from the official URL in a browser)


### Cell 3: Confirm the ingested inputs

In [3]:
# Cell 3: Verify inputs
files = sorted(f for f in RAW_DIR.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW_DIR), "bytes:", f.stat().st_size)
assert files, "Download or upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 3
act872_en.pdf bytes: 439389
kesuma_faq.pdf bytes: 686646
pua143_2026.pdf bytes: 269365
Upload verified. Select your relevant input(s) for Step A3.


### Step A3: Inspect the sources

In [4]:
# Step A3: Inspect sources
%pip install -q pypdf
from pypdf import PdfReader

def inspect_pdf(sid, pages_to_show):
    path = RAW_DIR / SOURCES[sid]["file"]
    reader = PdfReader(str(path))
    pages = [page.extract_text() or "" for page in reader.pages]
    meta = {k.lstrip("/"): str(v) for k, v in (reader.metadata or {}).items()}
    summary = {
        "source_id": sid,
        "pages": len(pages),
        "characters": sum(len(p) for p in pages),
        "empty_pages": sum(1 for p in pages if not p.strip()),
        "undecoded_chars_U+FFFD": sum(p.count("�") for p in pages),
        "legislation_no": meta.get("legislationNo", "not in metadata"),
        "publication_date": meta.get("publicationDate", "not in metadata"),
        "creator": meta.get("Creator", ""),
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    for n in pages_to_show:
        print(f"\n--- {sid} page {n} (first 400 characters) ---")
        print(pages[n - 1][:400])
    return pages, summary

act_pages, act_summary = inspect_pdf("S01", [2, 17])
reg_pages, reg_summary = inspect_pdf("S02", [2, 3])
faq_pages, faq_summary = inspect_pdf("S03", [2, 42])
inspection_summary = [act_summary, reg_summary, faq_summary]

Note: you may need to restart the kernel to use updated packages.
{
  "source_id": "S01",
  "pages": 59,
  "characters": 77702,
  "empty_pages": 0,
  "undecoded_chars_U+FFFD": 0,
  "legislation_no": "Act 872",
  "publication_date": "2025-12-31T00:00:00Z",
  "creator": "Adobe InDesign CS6 (Macintosh)"
}

--- S01 page 2 (first 400 characters) ---
2
Laws of Malaysia Act 872
Date of Royal Assent ... ... 16 December 2025
Date of publication in the  ... ... 31 December 2025
Gazette
Publisher’s Copyright   C
PERCETAKAN NASIONAL MALAYSIA BERHAD
All rights reserved. No part of this publication may be reproduced, stored in a retrieval system or transmitted in any form or by any means 
electronic, mechanical, photocopying, recording and/or otherwis

--- S01 page 17 (first 400 characters) ---
Gig Workers 17
 (3) The platform provider may modify or suspend the access 
of the gig worker to his digital intermediary system for a period 
not exceeding fourteen days for the purpose of holding an inquiry

**Observations**

1. All three are digital PDFs with no empty pages, so no OCR needed.
2. Act 872 page 2 has a publisher notice: all rights reserved.
3. Every Act page has a running header, and pages 1 to 8 are the cover, copyright page and Arrangement of Sections.
4. Text decodes fine. Definitions use curly quotes.
5. P.U.(A) 143/2026 has BM on page 2 and English on page 3, with some broken spacing ("t he").
6. The FAQ is BM only, with numbered questions grouped by topic.

## Part B: Prepare the Data

In [5]:
# Cell 4: Source selection and lineage utilities
import hashlib

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(p for p in RAW_DIR.rglob("*") if p.is_file()
                     and (filename is None or p.name == filename)
                     and (suffixes is None or p.suffix.lower() in suffixes))
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target.relative_to(BASE_DIR).as_posix())
    return manifest

print("Cell 4 ready")

Cell 4 ready


In [6]:
# Cell 4A: Lineage registry
lineage_entries = {}

def register_lineage(source, output, processing):
    source, output = Path(source).resolve(), Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip() or processing.upper().startswith("REPLACE"):
        raise ValueError("A real processing description is required.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    lineage_entries[(str(source), str(output))] = {"source": source, "output": output, "processing": processing.strip()}
    print("Registered:", source.name, "->", output.name)

print("Registered entries:", len(lineage_entries))

Registered entries: 0


### Cell 5A1: Clean the text and compare chunking

Cleaning: remove running headers, skip pages 1 to 8 (but read the section titles from them), change curly quotes to straight quotes, collapse spaces.

Comparing three approaches on the same text: fixed 500, paragraph aware 500 (lab code) and section aware (split by the Act's section numbers, max 1,200 characters).

In [7]:
# Cell 5A1: Extraction, cleaning and chunking
import re

ACT_HEADERS = [re.compile(r"^Gig Workers\s+\d+$"), re.compile(r"^\d+$"), re.compile(r"^Laws of Malaysia\s+Act 872$")]
TITLE_BLOCK = ["LAWS OF MALAYSIA", "Act 872", "GIG WORKERS ACT 2025"]
MAX_CHARS = 1200

def normalise_chars(text):
    return text.translate({0x201C: '"', 0x201D: '"', 0x2018: "'", 0x2019: "'"})

def clean_lines(pages, start_marker, headers=(), title_block=None, page_range=None):
    out, started = [], start_marker is None
    for page_no, raw in enumerate(pages, 1):
        if page_range and page_no not in page_range:
            continue
        lines = [re.sub(r"\s+", " ", ln).strip() for ln in normalise_chars(raw).split("\n")]
        while lines and (not lines[0] or any(p.match(lines[0]) for p in headers)):
            lines.pop(0)
        if title_block and lines[-len(title_block):] == title_block:
            lines = lines[:-len(title_block)]
        if not started:
            hits = [i for i, ln in enumerate(lines) if ln.startswith(start_marker)]
            if not hits:
                continue
            started, lines = True, lines[hits[0]:]
        out.extend((page_no, ln) for ln in lines if ln)
    return out

def toc_titles(pages, first, last):
    skip = re.compile(r"^(\d+|Section|Schedule|Part\s+[IVX]+|[A-Z ,'&]+|.*Laws of Malaysia.*|Gig Workers\s+\d+)$")
    titles, current = {}, None
    for ln in "\n".join(normalise_chars(pages[i - 1]) for i in range(first, last + 1)).split("\n"):
        ln = re.sub(r"\s+", " ", ln).strip()
        m = re.match(r"^(\d{1,3})\.\s+(.*)$", ln)
        if m:
            current = int(m.group(1))
            titles[current] = m.group(2).strip()
        elif current and ln and not skip.match(ln):
            titles[current] += " " + ln
    return titles

def words(text):
    return [w for w in re.sub(r"[^a-z0-9 ]", " ", text.lower()).split() if w not in {"the", "a", "an"}]

def split_sections(lines, n_expected, titles=None, unit="Section"):
    blocks = [{"section": "Preamble", "part": None, "lines": [], "pages": []}]
    part, expected, pending_part = None, 1, None
    for page, ln in lines:
        if re.match(r"^Part\s+[IVX]+$", ln):
            pending_part = ln
            continue
        if pending_part:
            part, pending_part = f"{pending_part} {ln}", None
            continue
        if expected <= n_expected and re.match(rf"^{expected}\.\s", ln):
            blocks.append({"section": str(expected), "part": part, "lines": [], "pages": []})
            expected += 1
        elif ln.upper() == "SCHEDULE" and expected > n_expected:
            blocks.append({"section": "Schedule", "part": "Schedule", "lines": [], "pages": []})
            continue
        blocks[-1]["lines"].append(ln)
        if page not in blocks[-1]["pages"]:
            blocks[-1]["pages"].append(page)
    moved = 0
    if titles:
        for i in range(len(blocks) - 1):
            nxt = blocks[i + 1]["section"]
            if not nxt.isdigit():
                continue
            target, body = words(titles.get(int(nxt), "")), blocks[i]["lines"]
            for k in (4, 3, 2, 1):
                if len(body) > k and words(" ".join(body[-k:])) == target:
                    del body[-k:]
                    moved += 1
                    break
    for b in blocks:
        b["title"] = titles.get(int(b["section"]), "") if titles and b["section"].isdigit() else b["section"]
        b["unit"] = unit
    return [b for b in blocks if b["lines"]], expected - 1, moved

FALLBACK_CUTS = []

def split_long(unit, max_chars):
    for pattern in (r" (?=\([a-z]{1,4}\) )", r"(?<=;) "):
        pieces = re.split(pattern, unit)
        if len(pieces) > 1:
            return pack(pieces, max_chars, joiner=" ")
    FALLBACK_CUTS.append(unit[:40])
    return [unit[i:i + max_chars] for i in range(0, len(unit), max_chars)]

def pack(units, max_chars, joiner="\n"):
    chunks, cur = [], ""
    for u in units:
        if len(u) > max_chars:
            if cur:
                chunks.append(cur)
                cur = ""
            chunks.extend(split_long(u, max_chars))
        elif len(cur) + len(u) + 1 <= max_chars:
            cur = f"{cur}{joiner}{u}" if cur else u
        else:
            chunks.append(cur)
            cur = u
    if cur:
        chunks.append(cur)
    return chunks

def section_units(block_lines):
    units, cur = [], []
    for ln in block_lines:
        if cur and re.match(r'^(\(\d+[A-Z]?\)|"[^"]+" (means|includes))', ln):
            units.append(" ".join(cur))
            cur = []
        cur.append(ln)
    if cur:
        units.append(" ".join(cur))
    return units

def section_chunks(blocks, max_chars=MAX_CHARS):
    out = []
    for b in blocks:
        pieces = pack(section_units(b["lines"]), max_chars)
        for k, piece in enumerate(pieces, 1):
            out.append({**{key: b[key] for key in ("section", "title", "part", "pages", "unit")},
                        "piece": k, "pieces": len(pieces), "text": piece})
    return out

def fixed_chunks(s, size=500):
    return [s[i:i + size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size=500):
    parts, current = [], ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

f = select_source(relative_path=SOURCES["S01"]["file"])
act_titles = toc_titles(act_pages, 3, 8)
act_lines = clean_lines(act_pages, "An Act to protect", ACT_HEADERS, TITLE_BLOCK)
act_blocks, act_found, headings_moved = split_sections(act_lines, 112, act_titles)
clean = "\n".join(ln for _, ln in act_lines)

print("Source:", f.name)
print("Pages:", len(act_pages), "| raw characters:", sum(map(len, act_pages)), "| cleaned characters:", len(clean))
print("Curly quotes left after cleaning:", sum(clean.count(q) for q in "“”"))
print("Official titles read from the Arrangement of Sections:", len(act_titles))
print("Sections detected in the body:", act_found, "| marginal headings moved back:", headings_moved)
assert clean.strip(), "No usable text: check extraction or OCR needs."

fixed = fixed_chunks(clean, 500)
structured = paragraph_chunks(clean, 500)
sectioned = section_chunks(act_blocks)
print("\nBlank-line paragraph breaks in the cleaned text:", len(re.findall(r"\n\s*\n", clean)))
print("Fixed chunks:", len(fixed), "| Paragraph-aware chunks:", len(structured), "| Section-aware chunks:", len(sectioned))
print("Paragraph-aware output identical to fixed output:", structured == fixed)

Source: act872_en.pdf
Pages: 59 | raw characters: 77702 | cleaned characters: 70116
Curly quotes left after cleaning: 0
Official titles read from the Arrangement of Sections: 112
Sections detected in the body: 112 | marginal headings moved back: 111

Blank-line paragraph breaks in the cleaned text: 0
Fixed chunks: 141 | Paragraph-aware chunks: 141 | Section-aware chunks: 134
Paragraph-aware output identical to fixed output: True


### Step B3: Comparison

In [8]:
# Step B3: Compare approaches
section_spans = []
pos = 0
for b in act_blocks:
    block_text = "\n".join(b["lines"])
    start = clean.find(block_text[:60], pos)
    section_spans.append((start, start + len(block_text), b["section"]))
    pos = start + 1

def sections_in(start, end):
    return {sec for s, e, sec in section_spans if s < end and e > start}

def evaluate_plain(chunks):
    mixed, boundary, offset = 0, 0, 0
    for c in chunks:
        start = clean.find(c, offset)
        offset = start + len(c)
        mixed += len(sections_in(start, offset)) > 1
        boundary += bool(re.match(r"^(\d{1,3}\.\s|\(\d+\)|\"[^\"]+\" means)", c.lstrip()))
    return mixed, boundary

PROBE = "not exceeding fourteen days for the purpose of holding an inquiry"

def probe_plain(chunks):
    hits = [c for c in chunks if PROBE in " ".join(c.split())]
    return f"whole phrase in one chunk: {bool(hits)}, section known: no (no metadata)"

def probe_sectioned(chunks):
    hits = [c for c in chunks if PROBE in c["text"]]
    return f"whole phrase in one chunk: {bool(hits)}, section known: {hits[0]['section'] if hits else 'n/a'}"

rows = []
for name, chunks in (("Fixed 500", fixed), ("Paragraph-aware 500", structured)):
    mixed, boundary = evaluate_plain(chunks)
    rows.append((name, len(chunks), round(sum(map(len, chunks)) / len(chunks)), mixed, boundary, "0%", probe_plain(chunks)))
BOUNDARY = r"^(\d{1,3}\.\s|\(\d+\)|\([a-z]{1,4}\)|\"[^\"]+\" means|\[|An Act)"
sec_boundary = sum(bool(re.match(BOUNDARY, c["text"])) for c in sectioned)
rows.append(("Section-aware", len(sectioned), round(sum(len(c["text"]) for c in sectioned) / len(sectioned)), 0,
             sec_boundary, "100%", probe_sectioned(sectioned)))

print(f"{'Approach':22}{'Chunks':>8}{'Avg chars':>11}{'Mixed':>8}{'Boundary':>10}{'Section id':>12}  Probe (Section 14(3))")
for r in rows:
    print(f"{r[0]:22}{r[1]:>8}{r[2]:>11}{r[3]:>8}{r[4]:>10}{r[5]:>12}  {r[6]}")

multi_piece = sorted({c["section"] for c in sectioned if c["pieces"] > 1}, key=lambda s: (not s.isdigit(), int(s) if s.isdigit() else 0))
print("\nSections split into more than one piece:", multi_piece)
print("Largest section-aware chunk:", max(len(c["text"]) for c in sectioned), "characters")
not_at_boundary = [(c["section"], c["piece"], c["text"][:50]) for c in sectioned if not re.match(BOUNDARY, c["text"])]
print("Section-aware chunks that do not start at a section, subsection, paragraph or definition:", not_at_boundary)
print("Fixed-size fallback cuts needed:", len(FALLBACK_CUTS))
print("\nFixed sample:           ", repr(fixed[33][:160]))
print("Section-aware sample:   ", repr(next(c for c in sectioned if c["section"] == "14")["text"][:160]))

Approach                Chunks  Avg chars   Mixed  Boundary  Section id  Probe (Section 14(3))
Fixed 500                  141        497      84         1          0%  whole phrase in one chunk: False, section known: no (no metadata)
Paragraph-aware 500        141        497      84         1          0%  whole phrase in one chunk: False, section known: no (no metadata)
Section-aware              134        494       0       134        100%  whole phrase in one chunk: True, section known: 14

Sections split into more than one piece: ['2', '8', '14', '25', '40', '42', '47', '75', '83', '84', '87', '90', '102', '109']
Largest section-aware chunk: 1194 characters
Section-aware chunks that do not start at a section, subsection, paragraph or definition: []
Fixed-size fallback cuts needed: 0

Fixed sample:            's Part\nshall be done in the manner as prescribed.\nConciliation by Minister\n20. (1) Notwithstanding any provision under this Part, the Minister\nmay conciliate an'
Section-aw

### Cell 5A2: Export section aware records (S01 and S02)

In [9]:
# Cell 5A2: Export records
CHOSEN_METHOD = "section-aware"
approach = f"section-aware chunks (max {MAX_CHARS} chars; long sections split at subsection/definition, then paragraph), headers and front matter removed, curly quotes normalised"

def build_records(chunks, sid, doc_label, cite_prefix, version):
    path = RAW_DIR / SOURCES[sid]["file"]
    digest = source_hash(path)
    records = []
    for i, c in enumerate(chunks, 1):
        unit = c["unit"]
        if c["section"].isdigit():
            cite = f"{cite_prefix}, {unit} {c['section']}"
        else:
            cite = f"{cite_prefix}, {c['section']}"
        label = f"{unit} {c['section']}: {c['title']}" if c["section"].isdigit() else c["section"]
        header = " | ".join(x for x in dict.fromkeys((doc_label, c["part"], label)) if x)
        if c["pieces"] > 1:
            header += f" (part {c['piece']} of {c['pieces']})"
        records.append({
            "record_id": f"{sid}-{c['section']}-{c['piece']:02d}",
            "source_id": sid,
            "cite": cite,
            "section": c["section"],
            "title": c["title"],
            "part": c["part"],
            "piece": c["piece"],
            "pieces": c["pieces"],
            "pages": c["pages"],
            "text": header + "\n" + c["text"],
            "char_count": len(c["text"]),
            "source": SOURCES[sid]["file"],
            "source_url": SOURCES[sid]["url"],
            "source_sha256": digest,
            "source_version": version,
            "language": "en",
            "processing": approach,
        })
    return records

act_records = build_records(sectioned, "S01", "Act 872 Gig Workers Act 2025", "Act 872", act_summary["publication_date"][:10])

reg_lines = clean_lines(reg_pages, "IN exercise of", page_range={3})
reg_lines = [(p, ln) for p, ln in reg_lines if not re.match(r"^(Made \d|\[KSM|DATO|Minister of Human)", ln)]
reg_titles = {1: "Citation and commencement", 2: "Percentage of contribution deduction", 3: "Notification by platform provider"}
reg_blocks, reg_found, _ = split_sections(reg_lines, 3, reg_titles, unit="Regulation")
reg_records = build_records(section_chunks(reg_blocks), "S02", "P.U.(A) 143/2026 Gig Workers (Social Security) Regulations 2026",
                            "P.U.(A) 143/2026", reg_summary["publication_date"][:10])
print("Regulations detected in S02:", reg_found)

def export(records, name):
    path = PROCESSED_DIR / name
    with path.open("w", encoding="utf-8") as out:
        for row in records:
            out.write(json.dumps(row, ensure_ascii=False) + "\n")
    back = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert len(back) == len(records), "Exported record count does not match"
    assert all(r["text"].strip() and r["cite"] for r in back), "Empty text or missing citation"
    return path

act_path = export(act_records, "prepared_act872_sections.jsonl")
reg_path = export(reg_records, "prepared_pua143_regulations.jsonl")
for path, recs in ((act_path, act_records), (reg_path, reg_records)):
    print(f"{path.name}: {len(recs)} records, {path.stat().st_size:,} bytes, Validation: PASS")

sample = next(r for r in act_records if r["section"] == "14")
print("\nSample record (Section 14, text shortened):")
print(json.dumps({**sample, "text": sample["text"][:220] + " ..."}, indent=2, ensure_ascii=False))
print("\nS02 Regulation 2:", next(r for r in reg_records if r["section"] == "2")["text"])

Regulations detected in S02: 3
prepared_act872_sections.jsonl: 134 records, 172,707 bytes, Validation: PASS
prepared_pua143_regulations.jsonl: 4 records, 3,953 bytes, Validation: PASS

Sample record (Section 14, text shortened):
{
  "record_id": "S01-14-01",
  "source_id": "S01",
  "cite": "Act 872, Section 14",
  "section": "14",
  "title": "Deactivation under service agreement, etc.",
  "part": "Part III GIG WORKER",
  "piece": 1,
  "pieces": 3,
  "pages": [
    16,
    17,
    18
  ],
  "text": "Act 872 Gig Workers Act 2025 | Part III GIG WORKER | Section 14: Deactivation under service agreement, etc. (part 1 of 3)\n14. (1) A platform provider shall have the right to deactivate the access of a gig worker to his d ...",
  "char_count": 1019,
  "source": "act872_en.pdf",
  "source_url": "https://www.mohr.gov.my/aktapekerjagig2025/assets/documents/Act%20872.pdf",
  "source_sha256": "632727cce625e8e3f9c8360d0c04a9ac3435cb8cd0bf5d0a0b67c77040d270ab",
  "source_version": "2025-12-31",
  "

### Step B3: Output checks

In [10]:
# B3: Output checks
for prepared_path in (act_path, reg_path):
    assert prepared_path.exists(), "Prepared output missing."
    assert prepared_path.stat().st_size > 0, "Prepared output is empty."
    print("Output:", prepared_path.name, "Size:", prepared_path.stat().st_size)

act_sections = [r["section"] for r in act_records if r["section"].isdigit()]
checks = {
    "all 112 sections present": sorted(set(map(int, act_sections))) == list(range(1, 113)),
    "every section title matches the Arrangement of Sections": all(r["title"] == act_titles[int(r["section"])] for r in act_records if r["section"].isdigit()),
    "every record has a citation and page": all(r["cite"] and r["pages"] for r in act_records + reg_records),
    "no record above the size limit": all(r["char_count"] <= MAX_CHARS for r in act_records + reg_records),
    "no undecoded U+FFFD characters": all("�" not in r["text"] for r in act_records + reg_records),
    "no section ends with the next section's heading": not any(
        words(act_blocks[i]["lines"][-1]) == words(act_titles.get(int(act_blocks[i + 1]["section"]), "#"))
        for i in range(len(act_blocks) - 1) if act_blocks[i + 1]["section"].isdigit()),
    "Section 14(3) fourteen-day rule is in a Section 14 record": any(PROBE in r["text"] and r["section"] == "14" for r in act_records),
    "1.25% contribution rate is in Regulation 2": any("one point two five percent" in r["text"] and r["section"] == "2" for r in reg_records),
    "record ids are unique": len({r["record_id"] for r in act_records + reg_records}) == len(act_records + reg_records),
    "every Act chunk starts at a section, subsection, paragraph or definition": all(
        re.match(BOUNDARY, r["text"].split("\n", 1)[1]) for r in act_records),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values()), "Content check failed"
print("Validation: PASS")

Output: prepared_act872_sections.jsonl Size: 172707
Output: prepared_pua143_regulations.jsonl Size: 3953
PASS all 112 sections present
PASS every section title matches the Arrangement of Sections
PASS every record has a citation and page
PASS no record above the size limit
PASS no undecoded U+FFFD characters
PASS no section ends with the next section's heading
PASS Section 14(3) fourteen-day rule is in a Section 14 record
PASS 1.25% contribution rate is in Regulation 2
PASS record ids are unique
PASS every Act chunk starts at a section, subsection, paragraph or definition
Validation: PASS


### Step B4: Lineage (Cells 6A and 6B)

In [11]:
# B4.1: Register lineage
register_lineage(source=select_source(relative_path=SOURCES["S01"]["file"]), output=act_path, processing=approach)
register_lineage(source=select_source(relative_path=SOURCES["S02"]["file"]), output=reg_path,
                 processing=approach + "; English page (page 3) only")

Registered: act872_en.pdf -> prepared_act872_sections.jsonl
Registered: pua143_2026.pdf -> prepared_pua143_regulations.jsonl


In [12]:
# Cell 6A: Generate manifest
assert lineage_entries, "No lineage entries registered."
VERSION = {"act872_en.pdf": act_summary["publication_date"][:10] + " (gazette publication date, PDF metadata)",
           "pua143_2026.pdf": reg_summary["publication_date"][:10] + " (gazette publication date, PDF metadata)"}
RIGHTS = {"act872_en.pdf": "Public official document. Publisher notice on page 2: all rights reserved (Percetakan Nasional Malaysia Berhad). Full prepared text is not published on GitHub.",
          "pua143_2026.pdf": "Public official document published by the Attorney General's Chambers. Treated the same as S01."}
URL = {v["file"]: v["url"] for v in SOURCES.values()}

lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source, output = entry["source"], entry["output"]
    assert source.is_file() and output.is_file() and output.stat().st_size > 0
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_url": URL[source.name],
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())).replace("\\", "/"),
        "prepared_sha256": source_hash(output),
        "prepared_records": sum(1 for _ in output.open(encoding="utf-8")),
        "processing_note": entry["processing"],
        "source_version": VERSION[source.name],
        "access_rights": RIGHTS[source.name],
        "status": "Prepared; quality review pending (Topic 4)",
    })
manifest = save_manifest(lineage_records)
print("COMMON LINEAGE MANIFEST GENERATED | Total lineage entries:", len(manifest["records"]))

Saved lineage: data/week3_manifest.json
COMMON LINEAGE MANIFEST GENERATED | Total lineage entries: 2


In [13]:
# Cell 6B: Verify manifest
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
saved_records = saved_manifest.get("records", [])
assert saved_records and len(saved_records) == len(lineage_entries)
for record in saved_records:
    source, output = RAW_DIR / record["source_filename"], BASE_DIR / record["prepared_output"]
    assert source.is_file() and output.is_file()
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"
print("COMMON LINEAGE VALIDATION: PASS | Verified records:", len(saved_records))

COMMON LINEAGE VALIDATION: PASS | Verified records: 2


## Part C: V1 Data Pipeline

In [14]:
# Cell 7: Pipeline utilities
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def rel(path):
    return str(Path(path).resolve().relative_to(BASE_DIR.resolve())).replace("\\", "/")

def resolve_source(relative_path):
    source = RAW_DIR / relative_path
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, sources, inspect_fn, prepare_fn, configuration):
    inputs = [Path(s) for s in sources]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")
    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")
    validation = {"output_count": len(outputs),
                  "output_sha256": [sha256_file(p) for p in outputs],
                  "output_bytes": [p.stat().st_size for p in outputs]}
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": rel(p), "url": URL[p.name], "sha256": sha256_file(p)} for p in inputs],
        "outputs": [rel(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": {p.name: VERSION[p.name] for p in inputs},
        "access_rights": {p.name: RIGHTS[p.name] for p in inputs},
    }
    manifest_file = output_dir / "pipeline_manifest.json"
    manifest_file.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", rel(manifest_file))
    return record

def register_and_run(kind, sources, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, sources, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready")

Cell 7 ready


### Cell 8A: PDF pipeline

Same configuration as Part B. Stops if the Act does not give exactly 112 sections or the regulations do not give 3.

In [15]:
# Cell 8A: PDF pipeline
SOURCES_A = [resolve_source(SOURCES["S01"]["file"]), resolve_source(SOURCES["S02"]["file"])]
CONFIG_A = {"method": "section-aware", "max_chars": MAX_CHARS,
            "act_expected_sections": 112, "regulations_expected": 3,
            "cleaning": ["normalise curly quotes", "remove running headers", "skip front matter", "collapse spaces"]}

def read_text_pages(path):
    pages = [p.extract_text() or "" for p in PdfReader(str(path)).pages]
    if not any(p.strip() for p in pages):
        raise ValueError(f"{path.name}: no extractable text. Scanned PDFs need OCR.")
    return pages

def prepare_one(path, cfg):
    pages = read_text_pages(path)
    if path.name == SOURCES["S01"]["file"]:
        titles = toc_titles(pages, 3, 8)
        lines = clean_lines(pages, "An Act to protect", ACT_HEADERS, TITLE_BLOCK)
        blocks, found, _ = split_sections(lines, cfg["act_expected_sections"], titles)
        if found != cfg["act_expected_sections"] or len(titles) != cfg["act_expected_sections"]:
            raise ValueError(f"Act structure check failed: {found} sections, {len(titles)} titles")
        meta = PdfReader(str(path)).metadata or {}
        return build_records(section_chunks(blocks, cfg["max_chars"]), "S01", "Act 872 Gig Workers Act 2025",
                             "Act 872", str(meta.get("/publicationDate", ""))[:10]), len(pages)
    lines = clean_lines(pages, "IN exercise of", page_range={3})
    lines = [(p, ln) for p, ln in lines if not re.match(r"^(Made \d|\[KSM|DATO|Minister of Human)", ln)]
    blocks, found, _ = split_sections(lines, cfg["regulations_expected"], reg_titles, unit="Regulation")
    if found != cfg["regulations_expected"]:
        raise ValueError(f"Regulation structure check failed: {found} regulations")
    meta = PdfReader(str(path)).metadata or {}
    return build_records(section_chunks(blocks, cfg["max_chars"]), "S02",
                         "P.U.(A) 143/2026 Gig Workers (Social Security) Regulations 2026",
                         "P.U.(A) 143/2026", str(meta.get("/publicationDate", ""))[:10]), len(pages)

def inspect_documents(inputs, cfg):
    report = []
    for src in inputs:
        pages = read_text_pages(src)
        report.append({"source": src.name, "pages": len(pages), "characters": sum(map(len, pages)),
                       "empty_pages": sum(1 for p in pages if not p.strip()),
                       "undecoded_chars": sum(p.count("�") for p in pages)})
    return report

def prepare_documents(inputs, out, cfg):
    target = out / "prepared_text.jsonl"
    mapping, all_records = [], []
    for src in inputs:
        records, n_pages = prepare_one(src, cfg)
        all_records.extend(records)
        mapping.append({"source": rel(src), "output": rel(target),
                        "records": len(records), "record_ids": f"{records[0]['record_id']} .. {records[-1]['record_id']}"})
    with target.open("w", encoding="utf-8") as handle:
        for record in all_records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    return [target], mapping, f"{cfg['method']} chunks, max {cfg['max_chars']} characters"

run_A = register_and_run("documents", SOURCES_A, inspect_documents, prepare_documents, CONFIG_A)
print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 2 | Outputs: 1
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.


### Cell 9: Repeatability check

In [16]:
# Cell 9: Repeatability check
assert PIPELINE_RUNS, "No pipeline runs registered. Run at least one selected Cell 8 pathway first."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first, second = entry["first"], entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Source-to-output mapping changed for {kind}"
    print(f"PASS: {kind}: source fingerprints, prepared outputs and mappings match")
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents | Inputs: 2 | Outputs: 1
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents: source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


### Cell 10: Evidence files for GitHub

The Act says all rights reserved, so the full text is not uploaded to GitHub. This writes evidence files instead: record id, citation, pages, size, SHA-256 of the text and an 80 character preview.

In [17]:
# Cell 10: Evidence files
PREVIEW_CHARS = 80

def evidence_file(full_path, evidence_path):
    rows = []
    for line in full_path.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        body = r["text"].split("\n", 1)[1]
        rows.append({k: r[k] for k in ("record_id", "source_id", "cite", "section", "title", "part", "piece", "pieces",
                                        "pages", "char_count", "source_url", "source_sha256", "source_version", "processing")}
                    | {"text_sha256": hashlib.sha256(r["text"].encode("utf-8")).hexdigest(),
                       "preview": body[:PREVIEW_CHARS] + (" ..." if len(body) > PREVIEW_CHARS else "")})
    evidence_path.parent.mkdir(parents=True, exist_ok=True)
    with evidence_path.open("w", encoding="utf-8") as out:
        for row in rows:
            out.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"{rel(evidence_path)}: {len(rows)} records (full file sha256 {sha256_file(full_path)[:16]}...)")

evidence_file(act_path, PROCESSED_DIR / "evidence" / "prepared_act872_sections.evidence.jsonl")
evidence_file(reg_path, PROCESSED_DIR / "evidence" / "prepared_pua143_regulations.evidence.jsonl")
evidence_file(PIPE_DIR / "documents" / "prepared_text.jsonl", PIPE_DIR / "documents" / "prepared_text.evidence.jsonl")

data/processed/evidence/prepared_act872_sections.evidence.jsonl: 134 records (full file sha256 55fa2cc3004e83c0...)
data/processed/evidence/prepared_pua143_regulations.evidence.jsonl: 4 records (full file sha256 1f16d09c6075c520...)
data/pipeline_v1/documents/prepared_text.evidence.jsonl: 138 records (full file sha256 7107d727ccc96875...)
